In [ ]:
# libraries
import os
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator
import time

#Visualizers
from yellowbrick.classifier import ClassificationReport
from yellowbrick.classifier import ClassPredictionError
from yellowbrick.classifier import ConfusionMatrix
from yellowbrick.classifier import ROCAUC
from yellowbrick.classifier import PrecisionRecallCurve
import matplotlib.pyplot as plt

#Metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics import hamming_loss
from sklearn.metrics import log_loss
from sklearn.metrics import zero_one_loss
from sklearn.metrics import matthews_corrcoef

#Classifiers
from sklearn.linear_model import SGDClassifier

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from keras import optimizers

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import joblib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df=pd.read_excel("/content/drive/MyDrive/PaperExtracciónDental/dataset_extraccion_up_low_bimax.xlsx")

In [ ]:
df.drop(columns=['Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48',"Extraccion / no extraccion inf","Extraccion / no extraccion sup", "Extraccion / no extraccion","Paciente","Extraccion tejidos blandos",'discrepancia total inferior', 'discrepancia total superior',"Clase real"],inplace=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
LabelEncoder_1=LabelEncoder()
df["Genero"]=LabelEncoder_1.fit_transform(df["Genero"])
df["Etnia"]=LabelEncoder_1.fit_transform(df["Etnia"])
df["Clasificación  esqueletica"]=LabelEncoder_1.fit_transform(df["Clasificación  esqueletica"])
df["Relación molar"]=LabelEncoder_1.fit_transform(df["Relación molar"])
df["Relacion canina"]=LabelEncoder_1.fit_transform(df["Relacion canina"])
df["Relación premolar"]=LabelEncoder_1.fit_transform(df["Relación premolar"])


In [ ]:
#df = df[df['Tipo de extraccióin global'] != 'Extracción inf.']


In [ ]:
df['Tipo de extraccióin global'] = df['Tipo de extraccióin global'].map({'Extracción bima': 1, 'No extracción': 0,"Extracción sup.":1,'Extracción inf.':1})

In [ ]:
Labels = df['Tipo de extraccióin global']
Features = df.drop(['Tipo de extraccióin global'],axis=1)
X_train, X_test, y_train, y_test = train_test_split(Features, Labels, test_size=0.2, stratify=Labels, random_state=42)


In [ ]:
scaler = StandardScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn import decomposition

pca = decomposition.PCA(n_components=0.96,svd_solver='full',tol=1e-4)
pca.fit(X_train)
X_train = pca.transform(X_train)
X_test=pca.transform(X_test)

In [ ]:
print('Train data shape:', X_train.shape)
print('Train labels shape:', y_train.shape)
print('Test data shape:', X_test.shape)
print('Test labels shape:', y_test.shape)

Train data shape: (400, 19)
Train labels shape: (400,)
Test data shape: (100, 19)
Test labels shape: (100,)


In [ ]:
#classes
classes = [0, 1]

#MLP

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline([    ('clf', MLPClassifier())])

parameters = {
'clf__hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
    'clf__activation': ['logistic', 'relu'],
    'clf__solver': ['adam', 'sgd'],
    'clf__learning_rate': ['constant', 'adaptive'],
}
grid_search = GridSearchCV(pipeline, parameters, cv=5)
grid_search.fit(X_train, y_train)


GridSearchCV(cv=5, estimator=Pipeline(steps=[('clf', MLPClassifier())]),
             param_grid={'clf__activation': ['logistic', 'relu'],
                         'clf__hidden_layer_sizes': [(50,), (100,), (50, 50),
                                                     (100, 50)],
                         'clf__learning_rate': ['constant', 'adaptive'],
                         'clf__solver': ['adam', 'sgd']})

In [ ]:
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print("Puntuación de validación cruzada media del mejor modelo:")
print(grid_search.best_score_)

Mejores parámetros encontrados:
{'clf__activation': 'logistic', 'clf__hidden_layer_sizes': (100, 50), 'clf__learning_rate': 'constant', 'clf__solver': 'adam'}
Puntuación de validación cruzada media del mejor modelo:
0.9175000000000001


#SGD

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline([    ('clf', SGDClassifier())])

parameters = {
    'clf__loss': ['hinge', 'log', 'modified_huber'],
    'clf__penalty': ['l2', 'l1', 'elasticnet'],
    'clf__alpha': [0.0001, 0.001, 0.01],
    'clf__max_iter': [1000, 2000, 3000],
}
grid_search = GridSearchCV(pipeline, parameters, cv=5)
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=Pipeline(steps=[('clf', SGDClassifier())]),
             param_grid={'clf__alpha': [0.0001, 0.001, 0.01],
                         'clf__loss': ['hinge', 'log', 'modified_huber'],
                         'clf__max_iter': [1000, 2000, 3000],
                         'clf__penalty': ['l2', 'l1', 'elasticnet']})

In [ ]:
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print("Puntuación de validación cruzada media del mejor modelo:")
print(grid_search.best_score_)

Mejores parámetros encontrados:
{'clf__alpha': 0.01, 'clf__loss': 'hinge', 'clf__max_iter': 2000, 'clf__penalty': 'l2'}
Puntuación de validación cruzada media del mejor modelo:
0.9125


#GB

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline([    ('classifier', GradientBoostingClassifier())])
parameters = {
 'classifier__n_estimators': [50, 100, 200],  # Number of boosting stages
    'classifier__learning_rate': [0.1, 0.05, 0.01],  # Learning rate
    'classifier__max_depth': [3, 5, 7],

    'classifier__random_state':[10,20,30,40,50]
    # Maximum depth of individual regression estimators
}
grid_search = GridSearchCV(pipeline, parameters, cv=5)
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('classifier',
                                        GradientBoostingClassifier())]),
             param_grid={'classifier__learning_rate': [0.1, 0.05, 0.01],
                         'classifier__max_depth': [3, 5, 7],
                         'classifier__n_estimators': [50, 100, 200],
                         'classifier__random_state': [10, 20, 30, 40, 50]})

In [ ]:
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print("Puntuación de validación cruzada media del mejor modelo:")
print(grid_search.best_score_)

Mejores parámetros encontrados:
{'classifier__learning_rate': 0.05, 'classifier__max_depth': 3, 'classifier__n_estimators': 200, 'classifier__random_state': 10}
Puntuación de validación cruzada media del mejor modelo:
0.8625


#DTC

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline([    ('clf', DecisionTreeClassifier())])

parameters = {
    'clf__criterion': ['gini', 'entropy'],
    'clf__max_depth': [None, 5, 10, 15],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4],
}
grid_search = GridSearchCV(pipeline, parameters, cv=5)
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('clf', DecisionTreeClassifier())]),
             param_grid={'clf__criterion': ['gini', 'entropy'],
                         'clf__max_depth': [None, 5, 10, 15],
                         'clf__min_samples_leaf': [1, 2, 4],
                         'clf__min_samples_split': [2, 5, 10]})

In [ ]:
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print("Puntuación de validación cruzada media del mejor modelo:")
print(grid_search.best_score_)

Mejores parámetros encontrados:
{'clf__criterion': 'entropy', 'clf__max_depth': 5, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 2}
Puntuación de validación cruzada media del mejor modelo:
0.805


#KNN

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import Normalizer, StandardScaler, MinMaxScaler, PowerTransformer, MaxAbsScaler, LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier()),

])

# Definir la cuadrícula de parámetros a explorar
param_grid = {
    'knn__n_neighbors': [1,2,3,4, 5,6, 7,10],
    'knn__weights': ['uniform', 'distance'],
    'knn__algorithm':["auto", "ball_tree", "kd_tree", "brute"],
    'scaler': [StandardScaler(), MinMaxScaler(),
 Normalizer(), MaxAbsScaler()],
 'knn__n_neighbors': [1, 3, 5, 7, 10],
 'knn__p': [1, 2],
 'knn__leaf_size': [1, 5, 10, 15],
}

grid_search =  GridSearchCV(pipeline, param_grid, cv=2).fit(X_train, y_train)


In [ ]:
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print("Puntuación de validación cruzada media del mejor modelo:")
print(grid_search.best_score_)

Mejores parámetros encontrados:
{'knn__algorithm': 'auto', 'knn__leaf_size': 1, 'knn__n_neighbors': 10, 'knn__p': 2, 'knn__weights': 'uniform', 'scaler': Normalizer()}
Puntuación de validación cruzada media del mejor modelo:
0.845


#RF

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline([    ('classifier', RandomForestClassifier())])
parameters = {
  'classifier__n_estimators': [50, 100, 150,200,250,300,400,500],
    'classifier__max_depth': [None, 10, 20,30,40],
    'classifier__min_samples_split': [2, 4, 6,8,10],
    'classifier__random_state':[10,20,30,40,50]
}
grid_search = GridSearchCV(pipeline, parameters, cv=5)
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('classifier',
                                        RandomForestClassifier())]),
             param_grid={'classifier__max_depth': [None, 10, 20, 30, 40],
                         'classifier__min_samples_split': [2, 4, 6, 8, 10],
                         'classifier__n_estimators': [50, 100, 150, 200, 250,
                                                      300, 400, 500],
                         'classifier__random_state': [10, 20, 30, 40, 50]})

In [ ]:
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print("Puntuación de validación cruzada media del mejor modelo:")
print(grid_search.best_score_)

Mejores parámetros encontrados:
{'classifier__max_depth': 10, 'classifier__min_samples_split': 4, 'classifier__n_estimators': 100, 'classifier__random_state': 10}
Puntuación de validación cruzada media del mejor modelo:
0.875


#ETC

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline([    ('classifier', ExtraTreesClassifier())])

parameters = {
    'classifier__n_estimators': [50, 100, 150,200,250,300],

    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 4, 6],
    'classifier__min_samples_split': [2,4,6,8,10],
    'classifier__random_state':[10,20,30,40,50]
}
grid_search = GridSearchCV(pipeline, parameters, cv=5)
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('classifier', ExtraTreesClassifier())]),
             param_grid={'classifier__max_depth': [None, 10, 20],
                         'classifier__min_samples_split': [2, 4, 6, 8, 10],
                         'classifier__n_estimators': [50, 100, 150, 200, 250,
                                                      300],
                         'classifier__random_state': [10, 20, 30, 40, 50]})

In [ ]:
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print("Puntuación de validación cruzada media del mejor modelo:")
print(grid_search.best_score_)

Mejores parámetros encontrados:
{'classifier__max_depth': 10, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 200, 'classifier__random_state': 10}
Puntuación de validación cruzada media del mejor modelo:
0.9099999999999999


#SVC

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC

model = SVC()

# Definir los parámetros a ajustar y sus rangos
param_grid = {
    'C': [0.1, 1, 10, 100],       # Parámetro de regularización
    'gamma': [1, 0.1, 0.01, 0.001],  # Parámetro del kernel 'rbf'
    'kernel': ['rbf'],            # Tipo de kernel a utilizar
}

# Configurar la búsqueda de hiperparámetros utilizando GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, verbose=2)

# Realizar la búsqueda de hiperparámetros en los datos de entrenamiento
grid_search.fit(X_train, y_train)

# Mostrar los mejores parámetros encontrados
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)

# Evaluar el modelo con los mejores parámetros en el conjunto de prueba
accuracy = grid_search.best_estimator_.score(X_test, y_test)
print(f"Precisión del modelo en conjunto de prueba: {accuracy:.4f}")


Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.0s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.0s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.0s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.0s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.0s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=   0.0s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=   0.0s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=   0.0s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=   0.0s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=   0.0s
[CV] END ......................C=0.1, gamma=0.01, kernel=rbf; total time=   0.0s
[CV] END ......................C=0.1, gamma=0.01